# Aula 02 · Detecção e correlação de eventos

**SRE AI Ops CoE · Banco do Brasil · 10/11/2026**

## 1. Da síntese à triagem

**O que muda da Aula 01 pra Aula 02?** Na Aula 01, o modelo organizava os registros do Pix num resumo. A gente abria as fontes pra conferir o que ele tinha escrito. Hoje vamos usar os mesmos dados para decidir se o caso entra numa fila de investigação. O Python faz a conta e aplica as regras; o LangGraph organiza a ordem desses passos.

| O mesmo caso | Na Aula 01, o que lemos | Na Aula 02, o que o programa faz |
|---|---|---|
| **80 falhas em 1.000 solicitações** | O relato da API (E02) já trazia os números. O modelo podia incluí-los no resumo. | Às 10h05, a janela de 10h00 a 10h05 fecha. O código lê as duas contagens em `metricas.json`, divide 80 por 1.000 e compara os 8% com o limiar de 1% do laboratório. A regra sinaliza o caso pra investigação. |
| **Uma cópia do alerta** | A instrução dizia ao modelo que E03 era cópia de E02. | A regra lê esse vínculo, identifica E03 como cópia e guarda o ID do original. As 80 falhas vêm da janela de métricas; a cópia do alerta não muda essa contagem. |

> **O que você vai poder testar:** consulte às 10h02. O relato da API já chegou, mas a janela de métricas ainda está aberta. Avance pra 10h05 e confira o que muda: agora o programa tem as contagens da janela fechada pra calcular a taxa e aplicar a regra.

**Você está no plantão.** Chegam relatos de falhas nas transferências Pix e um sinal de CPU alta no cache de marketing. Se você precisa decidir o que investigar primeiro, por onde começa?

<img src="projeto/assets/00-percurso-aula.png" alt="Partimos dos sinais do plantão, fazemos a triagem e chegamos a uma decisão de investigação sustentada por evidências." width="760">

[Fonte Mermaid do percurso](projeto/assets/00-percurso-aula.mmd)

Hoje vamos tomar uma decisão com essas evidências. A **triagem** é o trabalho de conferir o que chegou, medir o impacto e escolher quais sinais investigar juntos.

| Vamos consultar | O que você vai tentar responder |
|---|---|
| **10h02** | Já posso usar a contagem de falhas da janela de 10h00–10h05? |
| **10h05** | Qual situação merece investigação? Quais sinais entram no mesmo grupo? |
| **10h06** | Chega uma medição feita antes. O que muda na evidência e na decisão? |

**O que você vai produzir.** Um registro curto com a prioridade, os motivos, os sinais relacionados e uma pergunta em aberto. No código, esse registro se chama **dossiê**. Na Aula 03, ele será o ponto de partida para investigar a causa.

**Aula em 10/11/2026.** As três consultas reconstituem o incidente fictício de 03/11/2026, no fuso UTC−03:00.

<details>
<summary>Vocabulário para consultar durante o caso</summary>

| Conceito | Pergunta | Exemplo |
|---|---|---|
| Observação | O que foi registrado? | Oito conexões ocupadas e 35 pedidos aguardando. |
| Desvio | O que mudou em relação à referência? | A latência da jornada subiu. |
| Impacto | O que ocorreu com as solicitações? | Uma parte das transferências falhou. |
| Alerta | O que chamou atenção? | O relato de falhas da API. |
| Candidato | A regra encontrou motivo para investigar? | A taxa de falhas atingiu o limiar didático. |
| Correlação | Quais sinais podemos examinar juntos? | Falhas, espera e timeout na mesma jornada. |
| Causa | Que mecanismo explica o problema? | Exige confrontar hipóteses com outras evidências. |

</details>

**Antes de abrir os dados.** O cache e a transferência chamaram atenção no mesmo horário. Que informação você consultaria para decidir se os dois sinais devem entrar no mesmo grupo?

## 2. Conhecer o projeto

**Pergunta.** Onde ficam os dados, os cálculos e as decisões?

<img src="projeto/assets/02-insumos.png" alt="Eventos, métricas e contexto entram na triagem, que devolve decisão, motivos e lacunas." width="730">

[Mermaid editável](projeto/assets/02-insumos.mmd)

| Abra | Para encontrar |
|---|---|
| [evidencias.json](projeto/data/evidencias.json) | Onze observações com serviço, ocorrência e recebimento. |
| [metricas.json](projeto/data/metricas.json) | Sete janelas, cada uma com contagens e p95. |
| [dados.py](projeto/src/aiops/dados.py) e [contratos.py](projeto/src/aiops/contratos.py) | Leitura, recorte por horário e validação de campos. |
| [triagem.py](projeto/src/aiops/triagem.py) | Contexto da jornada, nós, critérios e montagem do grafo. |
| [test_triagem.py](projeto/tests/test_triagem.py) | Cenários que conferem as decisões. |

**Como usar essas fontes.** Se eu quero saber o que foi observado, leio os registros de `evidencias.json`. Se eu quero calcular a taxa de falhas entre 10h00 e 10h05, uso as contagens de `metricas.json`. Para decidir o que vai junto na investigação, abro `triagem.py`: ali estão a jornada conhecida do Pix e o vínculo que identifica a cópia do alerta. Por exemplo, o cache de marketing aparece entre os registros, mas fica separado porque não faz parte da jornada conhecida.

**Preparação.** Siga o [README](README.md#executar-no-vs-code), selecione `projeto/.venv/Scripts/python.exe` como kernel e execute as células em ordem. A triagem usa dados locais e roda sem credenciais.

**Célula Python · preparação.** `dados.py` lê os dois arquivos e `contratos.py` valida os campos. Confira quantas observações e janelas entraram.

In [ ]:
from datetime import timedelta
from statistics import median

import pandas as pd
from IPython.display import Image, display
from langsmith import tracing_context

from aiops.dados import PROJETO, carregar_evidencias, carregar_metricas
from aiops.triagem import (
    DUPLICATAS_CONHECIDAS, JORNADA_PIX, DossieTriagem, EstadoTriagem,
    correlacionar, deduplicar, depois_de_detectar, detectar, entrada_pix,
    horario, medir, montar_dossie, montar_grafo_triagem, priorizar, recortar,
)

evidencias = carregar_evidencias()
janelas = carregar_metricas()
print(f"{len(evidencias)} observações | {len(janelas)} janelas de métricas")

**Leitura do resultado.** Onze observações e sete janelas compõem o caso. A disponibilidade será avaliada a cada consulta. A preparação inicial baixa dependências, enquanto a execução do notebook trabalha com o ambiente instalado.

## 3. Ler o incidente

**Pergunta.** O que cada componente faz na transferência?

<img src="projeto/assets/01-jornada.png" alt="Gateway, API de transferências, consulta de limites e banco de contas. Cache de marketing fora da jornada." width="730">

[Mermaid editável](projeto/assets/01-jornada.mmd)

O **gateway** recebe o pedido. A **API** coordena a transferência. A **consulta de limites** verifica se há limite para a operação e acessa o **banco de contas**. O **pool** mantém conexões reutilizáveis para acessar o banco. Quando todas estão ocupadas, um pedido pode entrar na **fila** de espera. **Timeout** é o fim da espera permitida por alguma etapa.

Cada linha abaixo é uma observação identificada por um código, como E04. C01 e C02 identificam mudanças registradas nas observações. Um **trace**, como T01, descreve a passagem de uma requisição pelas etapas. A Aula 03 vai usar esse percurso para confrontar hipóteses.

**Célula Python · leitura.** Os objetos vieram de [evidencias.json](projeto/data/evidencias.json). Localize o relato de fila, a cópia do alerta e a CPU do cache. Esta tabela mostra o acervo completo para estudo. O grafo recebe um corte para decidir o que já podia conhecer.

In [ ]:
def tabela_eventos(registros):
    return pd.DataFrame([
        {"ID": e.id, "Serviço": e.servico,
         "Ocorreu": e.ocorrido_em.strftime("%H:%M"),
         "Chegou": e.recebido_em.strftime("%H:%M") if e.recebido_em else "Desconhecido",
         "Observação": e.observacao}
        for e in registros
    ])

with pd.option_context("display.max_colwidth", None):
    display(tabela_eventos(evidencias))

print("Serviços da jornada:", " → ".join(JORNADA_PIX))
print("Vínculos de duplicata fornecidos pelo caso:", DUPLICATAS_CONHECIDAS)

**Interpretação.** A fila de 35 pedidos é um sinal na consulta de limites. O serviço pode esperar por uma conexão antes de executar a consulta. Uma consulta de referência de 20 ms descreve uma observação do banco, com cobertura limitada.

## 4. Escolher o momento da análise

**Volte ao plantão.** Antes de calcular, precisamos responder se a contagem da janela já estava disponível às 10h02.

**Pergunta.** O que já poderíamos usar às 10h02?

<img src="projeto/assets/03-janela.png" alt="O relato E02 já chegou às 10h02. A janela de 10h00 a 10h05 só fecha às 10h05." width="730">

[Mermaid editável](projeto/assets/03-janela.mmd)

Uma **janela** é o intervalo usado para agregar as métricas. O **corte** é o horário em que consultamos as informações. A janela `[10h00, 10h05)` inclui 10h00 e termina antes de incluir 10h05. O laboratório considera o agregado disponível no fechamento. Já o relato de falhas E02 tem seu próprio horário de recebimento e sua descrição não informa a janela da série.

**Célula Python · recorte.** Em [triagem.py](projeto/src/aiops/triagem.py), `recortar()` usa `selecionar_evidencias()` e `selecionar_metricas()` de [dados.py](projeto/src/aiops/dados.py). Compare E02, que aparece nos dois cortes, com a janela iniciada às 10h00.

Repare em como a chamada funciona: `entrada_pix()` carrega as entradas e `recortar()` devolve só as atualizações do nó. A gente pode inspecionar esse retorno isoladamente. O grafo faz a mesma chamada e combina as atualizações com o estado para as próximas funções.

In [ ]:
entrada_02 = entrada_pix(horario("10:02"))
entrada_05 = entrada_pix(horario("10:05"))
recorte_02 = recortar(entrada_02)
recorte_05 = recortar(entrada_05)

display(pd.DataFrame([
    {"Consulta": hora,
     "Janelas fechadas": ", ".join(j.inicio.strftime("%H:%M") for j in r["fechadas"]),
     "E02 recebido": any(e.id == "E02" for e in r["disponiveis"])}
    for hora, r in [("10h02", recorte_02), ("10h05", recorte_05)]
]))

**Interpretação.** Às 10h02, duas janelas estão fechadas. Às 10h05, entram os totais da terceira. Experimente comparar o corte de 10h04 com o de 10h05. Mantenha o início da janela em 10h00.

## 5. Medir o impacto

**Pergunta.** Das solicitações da janela que fechou às 10h05, quantas falharam?

| Medida | Conta | Como ler |
|---|---|---|
| Taxa de falhas | 80 ÷ 1.000 = 8% | Solicitações que falharam naquela janela. |
| Margem do SLO | 100% − 99,9% = 0,1% | Fração de falha associada ao objetivo didático de sucesso. |
| Burn rate | 8% ÷ 0,1% = 80× | O ritmo observado é 80 vezes a margem de erro. |

O **SLI** é o indicador medido. Aqui usamos a proporção de sucesso na jornada e sua relação com a taxa de falha. O **SLO** é o objetivo escolhido. Para saber quanto do orçamento de erro de um período já foi consumido, precisaríamos das contagens de todo o período pertinente. [Google SRE Workbook](https://sre.google/workbook/alerting-on-slos/)

**Célula Python · cálculo.** Abra `medir()` em [triagem.py](projeto/src/aiops/triagem.py). Juntamos a entrada e as chaves produzidas por `recortar()`. Confira o denominador antes de interpretar a taxa.

In [ ]:
estado_05 = {**entrada_05, **recorte_05}
estado_05.update(medir(estado_05))
m = estado_05["medicao"]

print("Solicitações:", m.solicitacoes)
print("Falhas:", m.falhas)
print(f"Taxa de erro: {m.taxa_erro:.1%}")
print(f"Burn rate: {m.burn_rate:.0f}×")

**Interpretação.** O resultado descreve a janela inicial. Uma janela com zero solicitações deixaria a divisão indefinida. O motivo ficará no dossiê.

## 6. Construir a referência de latência

**Pergunta.** Como sabemos se 900 ms é alto para essa jornada?

O **p95** resume a cauda de latência. Aproximadamente 95% das solicitações na janela ficaram naquele tempo ou abaixo dele. Os p95 de 150 e 155 ms vêm de duas janelas anteriores ao aumento de falhas. Usamos a mediana desses dois números como referência didática. Recalcular o p95 combinado exigiria as latências das solicitações ou uma distribuição agregável.

O limiar é **duas vezes a referência**. A escolha é fácil de conferir. Em produção, escolher a referência exigiria considerar volume, horário e sazonalidade.

**Célula Python · referência.** `medir()` em [triagem.py](projeto/src/aiops/triagem.py) busca as janelas escolhidas na entrada. A comparação abaixo usa a mesma observação de 880 ms e troca somente as janelas da referência. As janelas degradadas de 900 e 950 ms servem ao experimento.

In [ ]:
inicio = horario("10:10")
corte = horario("10:15")
fixa = entrada_pix(corte, inicio_alvo=inicio)
contaminada = entrada_pix(
    corte, inicio_alvo=inicio,
    inicios_referencia=(horario("10:00"), horario("10:05")),
)
comparacao = []
for nome, entrada in [("Anterior ao aumento", fixa), ("Com janelas degradadas", contaminada)]:
    medicao = medir({**entrada, **recortar(entrada)})["medicao"]
    valores = [j.p95_ms for j in medicao.janelas_referencia]
    limiar = entrada["parametros"].fator_p95 * median(valores)
    comparacao.append({
        "Referência": nome, "p95 de origem": valores,
        "Mediana (ms)": medicao.p95_referencia_ms, "Limiar (ms)": limiar,
        "p95 observado (ms)": medicao.p95_ms, "Latência dispara?": medicao.p95_ms >= limiar,
    })
display(pd.DataFrame(comparacao))

**Interpretação.** O p95 continua em 880 ms. O limiar muda de 305 para 1.850 ms quando incorporamos a degradação à referência. A diferença vem da régua de comparação. Experimente explicar qual comparação você defenderia e de quais dados precisaria para calibrá-la.

## 7. Decidir se há candidato

**Agora a decisão começa a tomar forma.** Já temos uma medida de impacto e uma referência para comparar a latência. Vamos usar essas duas informações para decidir se a janela merece investigação.

**Pergunta.** Se a referência de latência faltar, ainda conseguimos investigar a taxa de falhas?

O detector compara cada critério disponível: taxa de erro ≥ 1% ou p95 ≥ 2 × referência. Um disparo abre um candidato. Com todos os critérios avaliados abaixo dos limiares, o resultado é sem desvio pelos critérios. Se nenhum disparou e faltou informação, o dossiê registra insuficiência.

| Falhas | Latência | Decisão |
|---|---|---|
| Disparou | Indisponível | Candidato com lacuna. |
| Abaixo do limiar | Indisponível | Dados insuficientes. |
| Abaixo do limiar | Abaixo do limiar | Sem desvio pelos critérios. |

**Célula Python · detecção.** Leia `detectar()` e `comparar()` em [triagem.py](projeto/src/aiops/triagem.py). Veja separadamente valor, limiar, resultado e motivo. Taxa e limiar de taxa aparecem como frações (0,08 = 8%).

Abra `detectar()` e procure as duas chamadas a `comparar()`. Cada comparação devolve resultado e motivo. Só depois a função monta a lista de gatilhos e escolhe o status que o roteamento usa.

In [ ]:
estado_05.update(detectar(estado_05))
display(pd.DataFrame([c.model_dump() for c in estado_05["criterios"]]))
print("Decisão:", estado_05["status"])

# Variação do exercício: retiramos só uma janela de referência da cópia da entrada.
sem_baseline = {
    **entrada_05,
    "janelas": [j for j in entrada_05["janelas"] if j.inicio != horario("09:55")],
}
estado_parcial = {**sem_baseline, **recortar(sem_baseline)}
estado_parcial.update(medir(estado_parcial))
estado_parcial.update(detectar(estado_parcial))
display(pd.DataFrame([c.model_dump() for c in estado_parcial["criterios"]]))
print("Decisão com lacuna:", estado_parcial["status"])
print("Lacunas:", *estado_parcial["medicao"].lacunas, sep="\n")

**Interpretação.** Sem baseline, 8% ainda dispara o critério de falhas. Experimente prever o status se a taxa for 0,1% e o p95 continuar indisponível para comparação. A triagem deve registrar o que ficou em aberto.

## 8. Tratar a duplicata

**Pergunta.** O segundo alerta de 80 falhas representa outras 80 falhas?

O cenário fornece um vínculo explícito entre a cópia E03 e o relato E02. `deduplicar()` separa a cópia do conjunto de sinais únicos e conserva sua referência. As falhas da jornada vieram de `metricas.json`. Repetir a mesma descrição em dois eventos não aumenta a contagem.

**Célula Python · deduplicação.** Em [triagem.py](projeto/src/aiops/triagem.py), confira o vínculo `original_id`. Experimente pensar no resultado se a cópia chegar e o original ainda estiver indisponível.

In [ ]:
estado_05.update(deduplicar(estado_05))
display(pd.DataFrame([
    {"Cópia": c.evidencia.id, "Original": c.original_id, "Classe": c.classe, "Motivo": c.motivo}
    for c in estado_05["classificacoes"]
]))
print("IDs únicos para a próxima etapa:", ", ".join(e.id for e in estado_05["unicos"]))

**Interpretação.** E03 conserva seu ID para conferência. Se o original ainda estiver indisponível, a cópia fica marcada para verificar e aguarda evidência de vínculo. A base permanece inteira.

## 9. Relacionar os sinais

**Já sabemos que vale investigar o Pix.** Agora precisamos decidir quais registros ajudam a investigar esse caso.

**Pergunta.** O que dá errado se o horário for o único critério?

<img src="projeto/assets/05-grupo.png" alt="Eventos da jornada seguem para investigação. Cópia, cache e parceiro têm tratamento separado." width="730">

[Mermaid editável](projeto/assets/05-grupo.mmd)

Correlacionar é propor uma associação explicável. A regra combina a janela de associação de eventos e os serviços conhecidos da jornada. No laboratório, essa faixa começa dez minutos antes da janela-alvo e vai até o corte. E01 registra a mudança no pool antes das falhas. E05 registra CPU alta em outro serviço. E08 é contexto para considerar o parceiro externo.

Uma política de **supressão** controla o que chega como notificação. Aqui produzimos uma classificação auditável e preservamos os registros para inspeção.

**Célula Python · comparação.** Primeiro use apenas horário nos sinais únicos. Depois use `correlacionar()`, em [triagem.py](projeto/src/aiops/triagem.py). Confira por que E05 troca de grupo.

In [ ]:
inicio_associacao = estado_05["inicio_alvo"] - timedelta(
    minutes=estado_05["parametros"].minutos_associacao_anteriores
)
so_por_tempo = {
    e.id for e in estado_05["unicos"]
    if inicio_associacao <= e.ocorrido_em <= estado_05["corte"]
}
estado_05.update(correlacionar(estado_05))
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame([
        {"ID": c.evidencia.id, "Serviço": c.evidencia.servico,
         "Entraria só pelo horário?": c.evidencia.id in so_por_tempo,
         "Classificação contextual": c.classe, "Motivo": c.motivo}
        for c in estado_05["classificacoes"]
    ]))

**Interpretação.** E05 entra pela proximidade de horário e fica fora do grupo pela topologia conhecida. Se um serviço tiver topologia desconhecida, sua relação fica a verificar. Se o mapa estiver desatualizado, a classificação pode precisar de revisão.

## 10. Justificar a prioridade

**Pergunta.** Por que o grupo Pix merece atenção agora?

O critério de falhas sinaliza impacto observado na jornada e dá prioridade **alta** de investigação. Um disparo apenas de p95 merece **revisar** o desvio com mais contexto. A referência de latência é comparativa e o SLO do exemplo é de sucesso. Esta fila trata um candidato da jornada Pix.

**Célula Python · fila.** `priorizar()` e `montar_dossie()` ficam em [triagem.py](projeto/src/aiops/triagem.py). Leia a justificativa ao lado das referências e lacunas. A decisão sobre a causa fica para investigação.

Compare os retornos: `priorizar()` escreve a fila com os IDs relacionados. `montar_dossie()` reúne também os critérios, as classificações e as lacunas. Essa diferença vai aparecer no exercício, quando você selecionar o que um assistente receberia.

In [ ]:
estado_05.update(priorizar(estado_05))
dossie_manual = montar_dossie(estado_05)["dossie"]
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame([item.model_dump() for item in dossie_manual.fila]))
print("Lacunas para investigação:", *dossie_manual.lacunas, sep="\n")

**Interpretação.** A fila usa a série para medir impacto e os eventos para sustentar o contexto. A redução do pool, a espera e o timeout são compatíveis com uma hipótese de contenção. Ainda precisamos examinar a sequência e a duração das etapas.

## 11. Organizar no LangGraph

**Você já fez a triagem com as funções Python.** Agora vamos ligar as mesmas etapas e conferir se o grafo produz a mesma decisão.

**Pergunta.** Onde o programa escolhe o caminho?

<img src="projeto/assets/06-processo.png" alt="Visão do processo, dos dados disponíveis ao dossiê para investigação." width="730">

[Mermaid editável](projeto/assets/06-processo.mmd)

<img src="projeto/assets/07-grafo.png" alt="Grafo completo. Um candidato passa por deduplicação, correlação e prioridade. Outros resultados vão ao dossiê." width="550">

[Mermaid editável](projeto/assets/07-grafo.mmd)

O **estado** guarda as entradas e as atualizações. Um **nó** é uma função Python que lê o estado e retorna campos novos ou atualizados. Uma **aresta** conecta etapas. A aresta condicional chama `depois_de_detectar()`, que decide se precisamos montar a fila. Em um recorte inválido, o programa informa um erro de validação.

### O que cada função faz no grafo

| Nó | O que lê | O que escreve para a próxima etapa |
|---|---|---|
| `recortar` | Eventos, janelas e horário de corte | Eventos recebidos e janelas fechadas |
| `medir` | Janela analisada e janelas de referência | Taxa de erro, p95, referência e lacunas |
| `detectar` | Medições e limiares | Critérios avaliados, gatilhos e status |
| `deduplicar` | Eventos recebidos e relações conhecidas entre cópias | Eventos únicos e classificação das cópias |
| `correlacionar` | Eventos únicos, tempo e mapa da jornada | Classificações, vínculos e motivos |
| `priorizar` | Gatilhos e evidências relacionadas | Fila com prioridade, justificativa e referências |
| `montar_dossie` | Medição, critérios, classificações e fila, quando houver | Dossiê de triagem com evidências e lacunas |

No arquivo [triagem.py](projeto/src/aiops/triagem.py), procure `montar_grafo_triagem()`. Na célula abaixo, reconstruímos a mesma montagem para ver onde cada função entra. Em `add_node("medir", medir)`, o primeiro argumento é o nome que aparece no grafo e o segundo é a função que roda. Em `add_edge("recortar", "medir")`, a saída do recorte passa a estar disponível quando `medir()` lê o estado.

A saída de `detectar()` inclui um `status`. A função `depois_de_detectar()` usa esse campo para escolher o próximo nó. Um candidato segue por deduplicação, correlação e prioridade. Os outros caminhos vão direto a `montar_dossie`. A função `configurar_fluxo_triagem()` monta as etapas comuns. Na demonstração, ligamos `montar_dossie` a `END`. No exercício, usamos esse ponto para acrescentar um nó.

Ao executar a célula, compare o percurso das 10h02 com o das 10h05. A coluna **Chaves atualizadas** vem de `stream()` e mostra o que cada função devolveu. Na Aula 03, um assistente pode partir das observações, das métricas e das lacunas já registradas. O corte e a seleção das evidências ficam sob regras que você consegue conferir aqui.

[Graph API do LangGraph](https://docs.langchain.com/oss/python/langgraph/graph-api)

In [ ]:
from langgraph.graph import END, START, StateGraph

def configurar_fluxo_triagem(esquema_estado=EstadoTriagem):
    montagem = StateGraph(esquema_estado)
    montagem.add_node("recortar", recortar)
    montagem.add_node("medir", medir)
    montagem.add_node("detectar", detectar)
    montagem.add_node("deduplicar", deduplicar)
    montagem.add_node("correlacionar", correlacionar)
    montagem.add_node("priorizar", priorizar)
    montagem.add_node("montar_dossie", montar_dossie)
    montagem.add_edge(START, "recortar")
    montagem.add_edge("recortar", "medir")
    montagem.add_edge("medir", "detectar")
    montagem.add_conditional_edges("detectar", depois_de_detectar, {
        "deduplicar": "deduplicar", "montar_dossie": "montar_dossie",
    })
    montagem.add_edge("deduplicar", "correlacionar")
    montagem.add_edge("correlacionar", "priorizar")
    montagem.add_edge("priorizar", "montar_dossie")
    return montagem


montagem = configurar_fluxo_triagem()
montagem.add_edge("montar_dossie", END)
grafo = montagem.compile()
assert grafo.get_graph().edges == montar_grafo_triagem().get_graph().edges

def executar(entrada):
    percurso, estado_final = [], {}
    # A sessão do notebook mantém a execução local mesmo com tracing no ambiente.
    with tracing_context(enabled=False):
        for modo, dado in grafo.stream(entrada, stream_mode=["updates", "values"]):
            if modo == "updates":
                percurso.extend({"Nó": no, "Chaves atualizadas": ", ".join(campos)}
                                for no, campos in dado.items())
            else:
                estado_final = dado
    return estado_final["dossie"], pd.DataFrame(percurso)

d_02, caminho_02 = executar(entrada_02)
d_05, caminho_05 = executar(entrada_05)
assert d_05 == dossie_manual

for d, caminho in [(d_02, caminho_02), (d_05, caminho_05)]:
    print(d.corte.strftime("%H:%M"), d.status)
    display(caminho)

**Interpretação.** Às 10h02, o grafo mede, detecta insuficiência e monta o dossiê. Às 10h05, executa também deduplicação, correlação e prioridade. O percurso registra as etapas efetivamente executadas.

## 12. Receber a evidência atrasada

**Pergunta.** O que muda quando uma observação anterior chega depois?

<img src="projeto/assets/04-chegada.png" alt="E09 ocorreu às 10h01 e só fica disponível no corte das 10h06." width="730">

[Mermaid editável](projeto/assets/04-chegada.mmd)

**Antes de executar.** Preveja se a chegada de E09 altera as referências, a taxa de falhas ou a prioridade. Na [atividade em grupo](#15-atividade-em-grupo--acrescente-um-nó-de-contexto-para-o-rca), você vai conferir se a leitura também entra no contexto preparado para o assistente.

**Célula Python · novo corte, mesma janela.** `entrada_pix()` em [triagem.py](projeto/src/aiops/triagem.py) mantém o início da janela às 10h00. A consulta avança até 10h06 e `recortar()` seleciona os eventos já recebidos. Abra a linha adicionada e leia os dois horários.

In [ ]:
d_06, _ = executar(entrada_pix(horario("10:06")))
ids_05 = {e.id for e in d_05.evidencias_recebidas}
novos = [e for e in d_06.evidencias_recebidas if e.id not in ids_05]
display(tabela_eventos(novos))
print("Medição da mesma janela preservada:", d_06.medicao == d_05.medicao)
print("Prioridade:", d_06.fila[0].prioridade)

**Interpretação.** E09 acrescenta uma observação do banco ocorrida às 10h01. A janela de falhas é a mesma. Uma amostra de CPU de 35% tem cobertura limitada para explicar a condição do banco ao longo do incidente.

## 13. Acompanhar a evolução

**Pergunta.** Como contar o que aconteceu depois sem reescrever o impacto da janela inicial?

<img src="projeto/assets/08-serie.png" alt="Taxas de erro e p95 das sete janelas, com limiares do laboratório." width="750">

Cada ponto mostra uma janela de cinco minutos. A janela iniciada às 10h20 termina às 10h25. Às 10h20 podemos avaliar a janela que começou às 10h15. A restauração do pool (E10) aconteceu às 10h12 e houve redução de falhas reportada às 10h15 (E11). A sequência ajuda a formular uma hipótese e orienta o que confrontar com outras fontes.

**Célula Python · dois alvos, mesmo corte.** Em [triagem.py](projeto/src/aiops/triagem.py), `entrada_pix()` aceita o horário da janela-alvo separadamente. Confira a referência de 152,5 ms nos dois casos.

In [ ]:
corte_20 = horario("10:20")
d_inicio, _ = executar(entrada_pix(corte_20))
d_recuperacao, caminho_recuperacao = executar(
    entrada_pix(corte_20, inicio_alvo=horario("10:15"))
)
display(pd.DataFrame([
    {"Janela": f"{d.medicao.inicio:%H:%M}–{d.medicao.fim:%H:%M}",
     "Corte": d.corte.strftime("%H:%M"), "Erro": f"{d.medicao.taxa_erro:.1%}",
     "Burn rate": f"{d.medicao.burn_rate:.0f}×", "p95 (ms)": d.medicao.p95_ms,
     "Referência (ms)": d.medicao.p95_referencia_ms, "Decisão": d.status}
    for d in [d_inicio, d_recuperacao]
]))
display(caminho_recuperacao)

**Interpretação.** A primeira janela continua em 8% e 80×. A posterior tem 0,2%, 2× e p95 de 170 ms. Os dois critérios ficam abaixo dos limiares de triagem na janela posterior. A margem associada ao SLO ainda é 0,1%. A melhora posterior orienta o RCA e o cumprimento do SLO no período completo exigiria outros dados.

## 14. Avaliar o detector

**Aprofundamento.** A decisão do Pix já está registrada. Agora vamos perguntar como comparar duas regras se esse tipo de triagem for usado em muitos casos.

**Pergunta.** Reduzir a quantidade de alertas é sempre uma melhoria?

Em um **conjunto sintético separado** do Pix, dez incidentes relevantes foram rotulados. Um verdadeiro positivo é um incidente relevante encontrado. Um falso positivo é um alerta sem incidente relevante associado. Um falso negativo é um incidente relevante perdido.

- **Precisão** responde “dos alertas emitidos, quantos eram relevantes?”.
- **Recall** responde “dos incidentes relevantes, quantos encontramos?”.

**Célula Python · cálculo local.** A tabela de entrada está na própria célula. Confira denominadores e compare a quantidade de alertas com a quantidade de incidentes perdidos. [Google SRE Workbook](https://sre.google/workbook/alerting-on-slos/)

In [ ]:
politicas = pd.DataFrame([
    {"Política": "A", "VP": 8, "FP": 12, "FN": 2},
    {"Política": "B", "VP": 7, "FP": 3, "FN": 3},
])
politicas["Alertas"] = politicas["VP"] + politicas["FP"]
politicas["Precisão (%)"] = 100 * politicas["VP"] / politicas["Alertas"]
politicas["Recall (%)"] = 100 * politicas["VP"] / (politicas["VP"] + politicas["FN"])
display(politicas)

**Interpretação.** A política B emite metade dos alertas e perde mais um incidente relevante. Considere o custo de investigar alertas adicionais e o custo de deixar um incidente passar. Num agrupamento, um falso vínculo é um erro de associação entre evidências. As métricas desta tabela se referem a incidentes detectados em outro conjunto.

## 15. Atividade em grupo · Acrescente um nó de contexto para o RCA

Imagine que um assistente de investigação precisa ler a triagem do Pix. O dossiê inteiro também guarda duplicatas, sinais de outros serviços e detalhes de auditoria. O assistente precisa de um recorte com observações identificáveis, o horário em que elas estavam disponíveis e as lacunas que a operação ainda precisa investigar.

**O que você vai construir.** Uma função chamada `preparar_contexto_rca` e um novo nó no LangGraph. Use o dossiê que o fluxo já produz e reaproveite os nós existentes. O resultado é um objeto que poderia entrar na etapa de investigação assistida por IA da Aula 03. Nesta atividade, você consegue inspecionar e testar esse contexto localmente, antes de enviar qualquer dado a um modelo.

```text
Fluxo de triagem → montar_dossie → preparar_contexto_rca → END
```

### 1. Escreva o nó

Abra a função abaixo. O grafo passa o estado como entrada. Leia `estado["dossie"]`. Quando a fila estiver vazia, devolva `{"contexto_rca": None}`. Se houver um item na fila, use `dossie.fila[0].referencias` para escolher as observações em `dossie.evidencias_recebidas`. A lista de referências já reflete o trabalho de deduplicação e correlação.

Devolva `{"contexto_rca": contexto}`. Use estas chaves no dicionário `contexto`:

| Chave | Conteúdo |
|---|---|
| `janela` | Lista com o início e o fim de `dossie.medicao`, em ISO com `.isoformat()` |
| `corte` | `dossie.corte` em ISO |
| `taxa_erro` | Proporção registrada na medição. Para 8%, mantenha `0.08`. |
| `evidencias` | Lista de objetos com `id`, `servico` e `observacao` |
| `lacunas` | Perguntas e limites já registrados no dossiê |

**Dica.** Transforme as referências do item da fila em um conjunto e percorra os eventos recebidos uma vez. Preserve o texto original de cada observação para que a pessoa consiga conferir a fonte.

In [ ]:
class EstadoComRCA(EstadoTriagem, total=False):
    contexto_rca: dict | None


def preparar_contexto_rca(estado: EstadoComRCA):
    # TODO: leia o dossiê e retorne o campo contexto_rca conforme o contrato.
    return {}

### 2. Ligue o nó e compile

`configurar_fluxo_triagem()`, da seção **Organizar no LangGraph**, monta todas as etapas até `montar_dossie`. Passe `EstadoComRCA` pra que o grafo possa guardar o novo campo. Complete as três linhas indicadas: registre a função com `add_node()` e ligue as duas arestas com `add_edge()`.

In [ ]:
def montar_grafo_com_rca():
    extensao = configurar_fluxo_triagem(EstadoComRCA)
    # TODO: adicione o nó "preparar_contexto_rca".
    # TODO: conecte "montar_dossie" ao novo nó.
    # TODO: conecte o novo nó a END.
    return extensao.compile()

### 3. Execute e confira

Troque `RODAR_EXERCICIO` para `True` depois de completar as duas células. A checagem passa o mesmo caso por três cortes. Às 10h02, a fila está vazia. Às 10h05, o contexto deve trazer as evidências da jornada e omitir a cópia E03, o cache E05 e a leitura E09 que ainda não chegou. Às 10h06, confira a entrada da leitura sem mudar a taxa da janela.

**Entrega do grupo.** Mostrem a nova ligação no grafo, o contexto gerado às 10h05 e os testes passando. Expliquem por que um assistente receberia esse recorte e como o ID permite conferir uma observação no dossiê.

In [ ]:
RODAR_EXERCICIO = False

if RODAR_EXERCICIO:
    grafo_com_rca = montar_grafo_com_rca()
    arestas_rca = {(a.source, a.target) for a in grafo_com_rca.get_graph().edges}
    assert ("montar_dossie", "preparar_contexto_rca") in arestas_rca
    assert ("preparar_contexto_rca", END) in arestas_rca

    with tracing_context(enabled=False):
        rca_02 = grafo_com_rca.invoke(entrada_pix(horario("10:02")))
        rca_05 = grafo_com_rca.invoke(entrada_pix(horario("10:05")))
        rca_06 = grafo_com_rca.invoke(entrada_pix(horario("10:06")))

    assert rca_02["contexto_rca"] is None, "Uma fila vazia deve produzir None."
    ctx_05, ctx_06 = rca_05["contexto_rca"], rca_06["contexto_rca"]
    assert rca_05["dossie"] == d_05, "O novo nó deve preservar a saída da triagem."
    assert ctx_05["janela"] == [horario("10:00").isoformat(), horario("10:05").isoformat()]
    assert ctx_05["corte"] == horario("10:05").isoformat()
    assert ctx_06["corte"] == horario("10:06").isoformat()
    assert ctx_05["taxa_erro"] == ctx_06["taxa_erro"] == 0.08
    assert ctx_05["lacunas"] == d_05.lacunas

    fontes = {e.id: (e.servico, e.observacao) for e in d_05.evidencias_recebidas}
    contexto_05 = {e["id"]: (e["servico"], e["observacao"]) for e in ctx_05["evidencias"]}
    assert contexto_05 == {id_: fontes[id_] for id_ in d_05.fila[0].referencias}
    assert len(contexto_05) == len(ctx_05["evidencias"])
    ids_06 = {e["id"] for e in ctx_06["evidencias"]}
    assert ids_06 == set(contexto_05) | {"E09"}

    print("Checagens passaram nos três cortes.")
    display(pd.DataFrame(ctx_05["evidencias"]))
else:
    print("Complete o nó e as conexões. Depois ative RODAR_EXERCICIO.")

### 4. Guarde o dossiê completo

A triagem ainda conserva o dossiê original. Rode a célula para conferir que a versão em JSON preserva todos os campos. Ela usa `DossieTriagem` de [triagem.py](projeto/src/aiops/triagem.py).

In [ ]:
dossie_json = d_inicio.model_dump_json()
dossie_reconstituido = DossieTriagem.model_validate_json(dossie_json)
assert dossie_reconstituido == d_inicio
print("Campos disponíveis ao RCA:", ", ".join(type(dossie_reconstituido).model_fields))
print("Pergunta aberta:", dossie_reconstituido.lacunas[0])

**Fechamento.** Você acrescentou ao fluxo uma etapa que prepara a entrada da investigação assistida por IA. Na Aula 03, vamos usar evidências e lacunas para comparar hipóteses. Uma delas é que parte do atraso ocorreu na fila de conexão.